# Imports

In [1]:
"""Run elasticity on the layered/faulted grid with hole."""

from pathlib import Path

from create_grid import create_layered_grid_with_hole
from common_functions import solve_and_export_pressure

#folder = Path(__file__).parent
folder = Path.cwd()



In [2]:


"""Run the layered-grid-with-hole elasticity workflow.

Parameters
----------
None
    The script uses fixed settings defined inside the function.

Returns
-------
None
    Solution files are written to the results directory.
"""
mesh_size = 0.025

# Step 1: generate layered/faulted grid with hole and layer id.
# Use function defaults for geometry, fault, and hole settings.
sd, cell_layer = create_layered_grid_with_hole(mesh_size=mesh_size, center=(1.5, 0.25))



c:\Users\steto\anaconda3.1\envs\poresct4amm2526\Lib\site-packages\porepy\grids\mdg_generation.py:924: UserWarning: Found 51 fractures outside the domain boundary
  warn(f"Found {num_missing} fractures outside the domain boundary")


In [3]:


# Step 2: solve elasticity and export displacement + layer id.
folder_export = folder / "results"

#give the desired pressure and the layer ID where you want to apply it
layer_ID = 3
pressure_value = 0.5

solve_and_export_pressure(
        sd,
        cell_layer,
        layer_ID,
        pressure_value,
        folder_export,
        export_name="sol_layered_ID3",
        cell_data=[("layer", cell_layer.astype(float))],
    )



array([[-0.00635488,  0.        ,  0.        , ...,  0.06347641,
         0.01835137,  0.07270313],
       [-0.0133904 ,  0.        ,  0.        , ..., -0.02008315,
         0.01390767, -0.00288462],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(3, 6256))

# Signature of the function

In [4]:
def GroupSolutionCP2( layer_ID , pressure_value, center_location_mean, center_location_std ):

    # your implementation

    # format of the output (as a function)
    # surface_displacement 
    #CI_corner = [(0.1,0.2),(0.15,0.25)]

    return mean_surface_displacement, CI_corner

In [5]:
import numpy as np
import joblib
import warnings
from scipy.interpolate import PchipInterpolator

# Silenziamo eventuali warning durante la fase di valutazione
warnings.filterwarnings("ignore")

def GroupSolutionCP2(layer_ID, pressure_value, center_location_mean, center_location_std):
    """
    Funzione di emulazione geomeccanica richiesta per la valutazione CP2.
    
    Parametri:
    - layer_ID (int): Identificativo del layer (0, 1, 2, 3).
    - pressure_value (float): Valore della pressione nell'intervallo [-0.5, 0.5].
    - center_location_mean (tuple): Coordinate medie (cx, cy) del centro dell'anomalia.
    - center_location_std (tuple): Deviazioni standard (std_x, std_y) ammesse tra [0.01, 0.045].
    
    Ritorna:
    - callable: Funzione che accetta una coordinata x in [0, 2] e restituisce [Ux, Uy].
    - list of tuples: Intervalli di confidenza al 90% per lo spostamento (Ux, Uy) nel punto (0, 1).
    """
    
    # 1. Validazione Input (Come richiesto dalle specifiche)
    std_x, std_y = center_location_std
    if not (0.01 <= std_x <= 0.045) or not (0.01 <= std_y <= 0.045):
        raise ValueError("Le deviazioni standard devono essere nell'intervallo [0.01, 0.045].")
    
    if not (-0.5 <= pressure_value <= 0.5):
        raise ValueError("La pressione deve essere nell'intervallo [-0.5, 0.5].")

    # 2. Caricamento dei Modelli GP
    try:
        # Assumiamo che la cartella 'models' sia nella directory corrente di esecuzione
        gp_ux_surf   = joblib.load(f"models/gp_ux_surf_layer_{layer_ID}.pkl")
        gp_uy_surf   = joblib.load(f"models/gp_uy_surf_layer_{layer_ID}.pkl")
        gp_corner_ux = joblib.load(f"models/gp_corner_ux_layer_{layer_ID}.pkl")
        gp_corner_uy = joblib.load(f"models/gp_corner_uy_layer_{layer_ID}.pkl")
    except FileNotFoundError:
        raise FileNotFoundError(f"Impossibile caricare i modelli per il layer {layer_ID}. Verificare la cartella 'models/'.")

    # 3. Predizione Profilo di Superficie (Nominale)
    # Calcoliamo il profilo esatto al centro medio per la funzione callable
    cx_mean, cy_mean = center_location_mean
    X_exact = np.array([[cx_mean, cy_mean, pressure_value]])
    
    exact_ux_surf = gp_ux_surf.predict(X_exact)[0]
    exact_uy_surf = gp_uy_surf.predict(X_exact)[0]
    
    # Creazione della funzione di interpolazione (x_eval ha 50 punti come nel training)
    x_eval = np.linspace(0.0, 2.0, 50)
    interp_ux = PchipInterpolator(x_eval, exact_ux_surf)
    interp_uy = PchipInterpolator(x_eval, exact_uy_surf)
    
    def mean_surface_displacement(x):
        """Restituisce lo spostamento [Ux, Uy] per una data coordinata x sulla superficie."""
        # Se x è fuori dai bordi fisici [0, 2], lo forziamo ai limiti per evitare crash di interpolazione
        x = np.clip(x, 0.0, 2.0)
        return [float(interp_ux(x)), float(interp_uy(x))]

    # 4. Quantificazione dell'Incertezza (UQ) per l'Angolo a (0,1)
    N_mc = 2000
    
    # Campionamento aleatorio
    cx_samples = np.clip(np.random.normal(cx_mean, std_x, N_mc), 0.1, 1.9)
    cy_samples = np.clip(np.random.normal(cy_mean, std_y, N_mc), 0.1, 0.9)
    p_samples = np.full(N_mc, pressure_value)
    
    X_pred = np.column_stack((cx_samples, cy_samples, p_samples))

    # Predizione dell'angolo con incertezza del modello (return_std=True)
    mean_cx, std_cx = gp_corner_ux.predict(X_pred, return_std=True)
    mean_cy, std_cy = gp_corner_uy.predict(X_pred, return_std=True)
    
    # Modello di Discrepanza: tolleranza del 5% + offset base per compensare l'errore del surrogato
    mesh_noise_x = 0.05 * np.abs(mean_cx) + 1e-4
    mesh_noise_y = 0.05 * np.abs(mean_cy) + 1e-4
    
    # Incertezza totale = Incertezza Epistemica GP + Discrepanza del Modello
    std_cx_adj = np.sqrt(std_cx**2 + mesh_noise_x**2)
    std_cy_adj = np.sqrt(std_cy**2 + mesh_noise_y**2)
    
    # Simulazione finale degli spostamenti (Aleatoria + Epistemica combinata)
    ux_corner_simulated = np.random.normal(mean_cx, std_cx_adj)
    uy_corner_simulated = np.random.normal(mean_cy, std_cy_adj)
    
    # 5. Formattazione dell'Output Finale (CI 90%)
    CI_corner = [
        (float(np.percentile(ux_corner_simulated, 5)), float(np.percentile(ux_corner_simulated, 95))),
        (float(np.percentile(uy_corner_simulated, 5)), float(np.percentile(uy_corner_simulated, 95)))
    ]

    return mean_surface_displacement, CI_corner